<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/8%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C1_%ED%95%9C%EA%B5%AD%EC%96%B4GPT%ED%85%8D%EC%8A%A4%ED%8A%B8%EC%83%9D%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8주차 과제 1: 한국어 GPT 텍스트 생성

다양한 시작 문장과 temperature로 텍스트를 생성하고 결과를 비교합니다.


In [ ]:
!pip install transformers


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="skt/kogpt2-base-v2",
)
print("모델 로드 완료!")


---
## 실험 1: Temperature별 비교


In [ ]:
# 같은 프롬프트에 temperature를 바꿔가며 결과 비교
prompt = "인공지능 기술이 발전하면"
print(f'프롬프트: "{prompt}"')
print("=" * 70)

for temp in [0.3, 0.7, 1.2]:
    print(f"\n[ Temperature = {temp} ]")
    results = generator(
        prompt,
        max_length=60,
        do_sample=True,
        temperature=temp,
        num_return_sequences=2,
    )
    for i, r in enumerate(results, 1):
        print(f"  생성{i}: {r['generated_text']}")


In [ ]:
# 두 번째 프롬프트로 Temperature 비교
prompt = "오늘 날씨가 좋아서"
print(f'프롬프트: "{prompt}"')
print("=" * 70)

for temp in [0.3, 0.7, 1.2]:
    print(f"\n[ Temperature = {temp} ]")
    results = generator(
        prompt,
        max_length=60,
        do_sample=True,
        temperature=temp,
        num_return_sequences=2,
    )
    for i, r in enumerate(results, 1):
        print(f"  생성{i}: {r['generated_text']}")


---
## 실험 2: 다양한 시작 문장 실험


In [ ]:
# 시작 문장에 따라 생성 결과가 어떻게 달라지는지 비교
prompts = [
    "대한민국의 수도는",
    "맛있는 음식을 먹으면",
    "프로그래밍을 배우려면",
    "행복한 삶을 위해서는",
    "여름 휴가를 계획한다면",
    "최근 가장 인기 있는",
]

print("=== 다양한 시작 문장 실험 (temperature=0.7) ===\n")

for prompt in prompts:
    result = generator(
        prompt,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
    )
    print(f'입력: "{prompt}"')
    print(f'생성: {result[0]["generated_text"]}')
    print("-" * 60)


---
## 실험 3: max_length에 따른 결과 비교


In [ ]:
# 생성 길이를 바꿔가며 결과 비교
prompt = "한국의 전통 음식 중에서"
print(f'프롬프트: "{prompt}"')
print("=" * 70)

for length in [30, 60, 100]:
    result = generator(
        prompt,
        max_length=length,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
    )
    text = result[0]["generated_text"]
    print(f"\nmax_length={length} (생성된 글자수: {len(text)}):")
    print(f"  {text}")


---
## 실험 4: do_sample 비교 (확률적 vs 결정적)


In [ ]:
# do_sample=False: 항상 가장 확률 높은 단어만 선택 (greedy)
# do_sample=True: 확률 분포에서 랜덤 샘플링 (다양한 결과)
prompt = "세상에서 가장 중요한 것은"
print(f'프롬프트: "{prompt}"')
print("=" * 70)

# Greedy (do_sample=False) — 매번 같은 결과
print("\n[ do_sample=False (Greedy) — 3번 실행해도 동일한 결과 ]")
for i in range(3):
    result = generator(
        prompt,
        max_length=50,
        do_sample=False,
        num_return_sequences=1,
    )
    print(f"  실행{i+1}: {result[0]['generated_text']}")

# Sampling (do_sample=True) — 매번 다른 결과
print("\n[ do_sample=True (Sampling, temp=0.7) — 매번 다른 결과 ]")
for i in range(3):
    result = generator(
        prompt,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
    )
    print(f"  실행{i+1}: {result[0]['generated_text']}")


---
## 실험 5: top_k / top_p 샘플링


In [ ]:
# top_k: 확률 상위 k개 단어 중에서만 선택
# top_p: 누적 확률이 p가 될 때까지의 단어 중에서 선택
prompt = "미래 사회에서는"
print(f'프롬프트: "{prompt}"')
print("=" * 70)

# top_k 비교
print("\n[ top_k 비교 ]")
for k in [5, 20, 50]:
    result = generator(
        prompt,
        max_length=50,
        do_sample=True,
        top_k=k,
        temperature=0.7,
        num_return_sequences=1,
    )
    print(f"  top_k={k:2d}: {result[0]['generated_text']}")

# top_p 비교
print("\n[ top_p 비교 ]")
for p in [0.5, 0.8, 0.95]:
    result = generator(
        prompt,
        max_length=50,
        do_sample=True,
        top_p=p,
        temperature=0.7,
        num_return_sequences=1,
    )
    print(f"  top_p={p}: {result[0]['generated_text']}")


---
## 실험 6: 같은 프롬프트로 여러 개 동시 생성


In [ ]:
# num_return_sequences로 한 번에 여러 결과를 비교
prompt = "좋은 프로그래머가 되려면"

results = generator(
    prompt,
    max_length=60,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=5,
)

print(f'프롬프트: "{prompt}"')
print(f"temperature=0.8 | 5개 동시 생성")
print("=" * 70)
for i, r in enumerate(results, 1):
    print(f"\n[{i}] {r['generated_text']}")


---
## 결과 정리

| 파라미터 | 값 | 효과 |
|---------|-----|------|
| temperature | 낮음(0.3) | 보수적, 반복적 |
| temperature | 높음(1.2) | 창의적, 엉뚱할 수 있음 |
| do_sample | False | 항상 같은 결과 (Greedy) |
| do_sample | True | 매번 다른 결과 (확률적) |
| top_k | 작을수록 | 선택지가 제한되어 보수적 |
| top_p | 작을수록 | 고확률 단어만 선택 |
| max_length | 길수록 | 더 긴 텍스트 생성 (뒤쪽 품질 저하 가능) |
